# Task 2 — Teaching a computer to read sentiment

In the first activity you sorted film reviews into **positive** and **negative** by hand, using lists of "good" and "bad" words.

Now we'll get a computer to do the same job — in **two different ways**:

1. **The rule-based model** — the method you used earlier.
2. **The AI model** — a small language model that understands *meaning*, not just words.

Then we'll see where each one succeeds and where it gets fooled.

---

### How to use this notebook
- Run each grey **code cell** by clicking it and pressing **▶** (or `Shift + Enter`).
- Run the cells **in order, top to bottom.**
- You don't need to understand every line. The parts you'll actually change are clearly marked with
  `# 👉 CHANGE THIS`.
- If you see a red error, try to re-run the cell above it, then try again.

## Step 0 — Install the AI library

This downloads the tools we need. It takes about **1–2 minutes** the first time.
You only need to run it once per session. A few warning messages are normal.

In [ ]:
# Installs the sentence-transformers library (for the AI model)
!pip install -q sentence-transformers

print("\n✅ Done! The library is installed. Move on to Step 1.")

## Step 1 — Model 1: the rule-based classifier

This is what we did on paper earlier. We give the computer two lists —
positive words and negative words — and it decides by counting which kind appears more.

Run the cell below to build it.

In [ ]:
# Our two word lists
positive_words = [
    "good", "great", "brilliant", "amazing", "wonderful", "love", "loved",
    "funny", "clever", "beautiful", "excellent", "perfect", "enjoyed",
    "gripping", "superb", "heartwarming", "exciting", "best", "recommend"
]

negative_words = [
    "bad", "boring", "terrible", "awful", "dull", "predictable", "weak",
    "poor", "dreadful", "disappointing", "forgettable", "slow", "mess",
    "waste", "lifeless", "lazy", "worst", "hate", "hated"
]

def rule_based_sentiment(review):
    """Counts positive vs negative words and picks the bigger side."""
    text = review.lower()                 # ignore capital letters
    words = text.split()                  # break the sentence into words
    pos_count = sum(word.strip(".,!?") in positive_words for word in words)
    neg_count = sum(word.strip(".,!?") in negative_words for word in words)

    if pos_count > neg_count:
        return "POSITIVE"
    elif neg_count > pos_count:
        return "NEGATIVE"
    else:
        return "UNSURE"

print("✅ Rule-based model ready.")

### Try it out

Run this cell to test the rule-based model on a review.
Then **change the sentence** and run it again to see what happens. Try a few different sentences and try to break the rule based method.

In [ ]:
review = "A brilliant film with wonderful acting"   # 👉 CHANGE THIS

result = rule_based_sentiment(review)
print(f'Review:  "{review}"')
print(f'Verdict: {result}')

## Step 2 — Model 2: the AI model

The rule-based model only knows the exact words on its lists. It has no idea that
*"fantastic"* is positive if we forgot to add it, and it can't tell that *"not good"* is negative.

Our second model is different. It's a small **AI language model** that turns any sentence into a list of numbers — a
**"fingerprint" of its meaning** — and sentences with similar meanings get similar fingerprints.

Run the cell below to load it. This takes about **30 seconds** the first time (it downloads the model).

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load the small, fast AI model (about 90 MB).
print("Loading the AI model... (this takes a moment)")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ AI model loaded and ready.")

### How the AI model decides

We give it a few **example** positive sentences and a few **example** negative ones.
To classify a new review, the model checks: *does this review's meaning look more like
the positive examples, or the negative ones?*

This is the AI version of "which pile does it belong in?" — but based on **meaning**, not word-matching.

Run this cell to set up the examples.

In [ ]:
# A few clear examples of each sentiment. The AI compares new reviews against these.
positive_examples = [
    "I loved this film, it was fantastic",
    "A wonderful, moving and beautiful movie",
    "Brilliant and enjoyable from start to finish",
]
negative_examples = [
    "I hated this film, it was rubbish",
    "A dull, disappointing and boring movie",
    "Terrible and a complete waste of time",
]

# Turn every example into its "meaning fingerprint" (a vector of numbers).
positive_vectors = model.encode(positive_examples)
negative_vectors = model.encode(negative_examples)

def ai_sentiment(review):
    """Compares the review's meaning to the positive and negative examples."""
    review_vector = model.encode(review)

    # How similar is this review to the positive examples? And the negative ones?
    pos_similarity = util.cos_sim(review_vector, positive_vectors).mean().item()
    neg_similarity = util.cos_sim(review_vector, negative_vectors).mean().item()

    if pos_similarity > neg_similarity:
        return "POSITIVE"
    else:
        return "NEGATIVE"

print("✅ AI classifier ready.")

### Try the AI model

Run this cell, then **change the sentence** and try your own. Run 10-15 samples through it: can you find out where it works well, and what patterns tend to break it?

In [ ]:
review = "Not bad, I enjoyed it more than I expected"   # 👉 CHANGE THIS

result = ai_sentiment(review)
print(f'Review:  "{review}"')
print(f'Verdict: {result}')

## Step 3 — Head to head

Let's run **both models on the same reviews** and put their
answers side by side.

Watch especially the **tricky** reviews at the bottom — the ones with *"not"*, sarcasm,
or words that aren't on our lists. Where do the two models disagree? Which one gets it right?

In [ ]:
test_reviews = [
    # Straightforward ones
    "A brilliant and wonderful film",
    "A boring and terrible waste of time",
    # Words NOT on the rule-based lists
    "An absolute masterpiece, utterly magnificent",
    "Tedious, clunky and instantly forgettable rubbish",
    # The tricky ones from your paper activity
    "This was not good at all",
    "Not bad actually, I really enjoyed it",
    "Oh great, another two hours I will never get back",
    "Far from boring, it was thrilling",
]

# Print a tidy comparison table.
print(f'{"REVIEW":<52} {"RULE-BASED":<12} {"AI MODEL":<10}')
print("-" * 76)
for review in test_reviews:
    r = rule_based_sentiment(review)
    a = ai_sentiment(review)
    short = (review[:49] + "...") if len(review) > 52 else review
    print(f'{short:<52} {r:<12} {a:<10}')

### What to look for

Look down the two columns and find the rows where the models **disagree**.

- **"An absolute masterpiece..."** — the rule-based model says `UNSURE`, because none of those
  words are on its lists! The AI still gets it, because it understands the *meaning*.
- **"This was not good at all"** — the rule-based model sees the word *"good"* and votes
  `POSITIVE`. It can't understand the word *"not"*. Does the AI do better?
- **"Oh great, another two hours I will never get back"** — pure sarcasm. There are no negative
  words at all. This one is genuinely hard — even the AI may get it wrong!

**Discussion:** Neither model is perfect. The rule-based one is easy to understand but brittle.
The AI is more flexible but harder to explain, and it still struggles with sarcasm. Real-world
sentiment tools face exactly these trade-offs.

## Step 4 — Your challenge

Now it's your turn. In the cell below, **write your own review** and see if you can:

1. Find a sentence where the two models **disagree**.
2. Find a sentence that **fools the AI** (makes it give the wrong answer).
3. Try sarcasm, slang, or emoji. What breaks it?

Change the sentence, run the cell, and keep experimenting.

In [ ]:
your_review = "Type your own film review here!"   # 👉 CHANGE THIS

print(f'Your review: "{your_review}"')
print(f'Rule-based says: {rule_based_sentiment(your_review)}')
print(f'AI model says:   {ai_sentiment(your_review)}')